## Course Announcements

Due Sunday (11:59 PM):
- Q7
- [weekly project survey](https://forms.gle/pRqEYm8rZQH4Ezcp7) (for EC) - complete weekly

Notes:
- **A6**: now due *next* Sun 11/22 (this is a change!); No A7 - everyone will get full credit for A7; Canvas/syllabus updated
    - I still recommend doing A6 this weekend; next weekend your EDA checkpoint is due 
- Section next week: Project-focused
- Lecture next *Friday* will be pre-recorded \[to get lecture credit, there will be a Google form to complete\]

# Machine Learning in Python

Today's question: **how do we train a model, check it honestly, and tune it, using `scikit-learn`?**

- The steps: **split → fit → predict → evaluate**
    - lock away a **test set**; make choices with a **validation set**
    - training vs. validation accuracy
    - `classification_report` & the confusion matrix
- **Scaling** for distance-based models like KNN (fit on the training data only)
- **Pipelines**: chaining preprocessing and a model together
- Tuning **hyperparameters** and **comparing models** with **cross-validation** (`GridSearchCV`), never with the test set
- **One-hot encoding** for categorical features
- A template for your project: `ColumnTransformer` + `Pipeline` + `GridSearchCV`, then scoring the final model on the test set, **once**

<div class="alert alert-info">

**A6:** This is the code for A6 Parts 2–4, with one difference in how we split the data; see [3a](#s3a).

</div>

The concepts are in the machine learning slides from the last two lectures.

On your own (end of notebook): decision trees, how noisy one validation set is, splitting by time, and how SVMs work.

For more reading on `scikit-learn`: https://scikit-learn.org/stable/index.html

## Contents

Reminder: In JupyterLab, the table of contents sidebar, the bulleted-list icon on the left, shows the same outline and highlights the section you're in.

- [1. Setup](#s1)
- [2. The Data: Penguins](#s2)
- [3. Split, Fit, Predict, Evaluate](#s3) · [3a. Lock Away a Test Set, Then Split Off a Validation Set](#s3a) · [3b. Fit & Predict](#s3b) · [3c. Evaluate](#s3c)
- [4. Scaling: Units Matter for KNN](#s4) · [4a. Look at the Scales](#s4a) · [4b. Fit the Scaler on Training Data Only](#s4b) · [4c. KNN on Scaled Data](#s4c)
- [5. Pipelines, Tuning & Comparing Models](#s5) · [5a. Pipelines](#s5a) · [5b. Training, Validation & Test](#s5b) · [5c. Tuning k with GridSearchCV](#s5c) · [5d. Comparing Different Models](#s5d)
- [6. Categorical Features: One-Hot Encoding](#s6)
- [7. A Template for Your Project](#s7) · [7a. Open the Test Set, Once](#s7a)
- [8. Summary: The Steps](#s8)
- [9. On Your Own](#s9) · [9a. A Different Model: Decision Trees](#s9a) · [9b. How Noisy Is One Validation Set?](#s9b) · [9c. How Should You Split? Time Matters](#s9c) · [9d. SVMs: Maximizing the Margin](#s9d) · [9e. Kernel SVMs](#s9e) · [9f. Softening the Margin: C](#s9f)

## 1. Setup <a id="s1"></a>

In [ ]:
import pandas as pd
import numpy as np

# Import seaborn and apply its plotting styles
import seaborn as sns
sns.set_theme(style="white",
              font_scale=2,
              rc={'axes.spines.right': False,'axes.spines.top': False}) # this last thing is equivalent to always having sns.despine()

# import matplotlib
import matplotlib.pyplot as plt
# set plotting size parameter
plt.rcParams['figure.figsize'] = (8, 8)

# scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

#improve resolution
%config InlineBackend.figure_format ='retina'

**The results log.** Every time we fit a model, or change something about it, we'll add a row to a running table: what changed, what data it was scored on, and how accuracy changed. *Change* is measured against the run in the *vs. run* column, which is always a run scored on the *same* data (training, validation, cross-validation, or test): a training score and a test score aren't a fair comparison, so the table never compares them. By default it's the previous run on that data.

Run the next cell to set it up. You don't need to read its code, but you can expand it if you're curious.

In [ ]:
# a running log of every model we fit today
results_log = []

# add a row and return its run number; the change in accuracy is relative to run number `compare_to`,
# or by default to the last run scored on the same data
def log_run(section, model, features, preprocessing, change, scored_on, accuracy, compare_to=None):
    if compare_to is None:
        same_data = [r['Run'] for r in results_log if r['Scored on'] == scored_on]
        compare_to = same_data[-1] if same_data else None
    if compare_to is not None:
        baseline = results_log[compare_to - 1]
        assert baseline['Scored on'] == scored_on, 'only compare runs scored on the same data'
    results_log.append({'Run': len(results_log) + 1,
                        'Section': section,
                        'Model': model,
                        'Features': features,
                        'Preprocessing': preprocessing,
                        'What changed': change,
                        'Scored on': scored_on,
                        'Accuracy': accuracy,
                        'Change': accuracy - baseline['Accuracy'] if compare_to else np.nan,
                        'vs. run': compare_to})
    return len(results_log)

# show the log, highlighting the newest `new` rows
def show_log(new=1):
    log = pd.DataFrame(results_log).set_index('Run')
    def highlight(row):
        style = 'background-color: #fff3cd; color: black' if row.name > len(log) - new else ''
        return [style] * len(row)
    return (log.style
               .apply(highlight, axis=1)
               .format({'Accuracy': '{:.3f}',
                        'Change': lambda d: '' if pd.isna(d) else f'{d:+.3f}',
                        'vs. run': lambda r: '' if pd.isna(r) else f'{r:.0f}'}))

# a short label like 'KNN (k=5)' from a model and its tuned hyperparameters
def describe(model, params):
    names = {'KNeighborsClassifier': 'KNN', 'SVC': 'SVM', 'DecisionTreeClassifier': 'Decision tree'}
    settings = []
    for key, value in params.items():
        if key == 'model':
            continue
        key = key.split('__')[-1].replace('n_neighbors', 'k')
        settings.append(f'{key}={value:.3g}' if isinstance(value, float) else f'{key}={value}')
    return f"{names.get(type(model).__name__, type(model).__name__)} ({', '.join(settings)})"

## 2. The Data: Penguins <a id="s2"></a>

Body measurements of 333 penguins from three species on three islands in the Palmer Archipelago, Antarctica. Our task: **predict a penguin's species from its body measurements.**

In [ ]:
# lets get some 🐧 data to work with
penguins = sns.load_dataset('penguins').dropna()
penguins.head()

In [ ]:
with sns.plotting_context('notebook'):
    sns.pairplot(penguins, hue='species')
plt.show()

The species separate pretty well in most of these plots, so this should be a reasonable prediction task.

For now we'll use the four numeric measurements as our features (`X`) and species as our outcome (`y`).

In [ ]:
features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

X = penguins[features]
y = penguins['species']

y.value_counts()

Note that the classes aren't balanced: there are about half as many Chinstraps as Adelies. Keep that in mind when we look at performance by species.

## 3. Split, Fit, Predict, Evaluate <a id="s3"></a>

<div class="alert alert-info">

**A6:** These are the same steps as A6 Part 2, with one difference in how we split the data (see the note in [3a](#s3a)).

</div>

### 3a. Lock Away a Test Set, Then Split Off a Validation Set <a id="s3a"></a>

Split first, before doing anything else. We split twice:

1. **Lock away a test set** (20% of the penguins). We won't look at it until we've made every decision, at the very end of the lecture ([7a](#s7a)).
2. **Split what's left into training and validation.** We fit models on the training set and compare our choices (which model, whether to scale, which `k`) on the validation set.

In [ ]:
# step 1: lock away 20% as the test set; don't touch it until the very end
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, random_state=30)

# step 2: split the rest into training (75%) and validation (25%), the same settings as A6
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, random_state=42)

print('training:  ', X_train.shape)
print('validation:', X_val.shape)
print('test:      ', X_test.shape, '(locked away)')

<div class="alert alert-info">

**How this differs from A6, and why.** A6 splits the data only once, into training (75%) and test (25%), and it compares options (raw vs. scaled features, with and without PCA) on that test set. That keeps the assignment short and focused on the mechanics: fitting, predicting, and reading the results.

The catch: every time you use the test set to make a choice, the test score becomes a little less honest, because the option you picked was chosen partly for doing well on those particular penguins. It's no longer data the final model has never influenced. So in lecture, and in your project, we make choices with **validation data** and score on the test set **once**, at the end.

The code is the same. Step 2 above is exactly A6's split; we just call the held-out 25% the *validation* set. Where A6 has `X_test` and `y_test`, read `X_val` and `y_val` here.

</div>

### 3b. Fit & Predict <a id="s3b"></a>

Every `scikit-learn` model works the same way: create it (choosing its hyperparameters), `.fit()` it on training data, then `.predict()`.

We'll use **k-nearest neighbors (KNN)** with `k = 1`: each penguin gets the species of the single most similar penguin in the training data.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_train, y_train)

y_train_pred = knn.predict(X_train)
y_val_pred = knn.predict(X_val)

print(f'training accuracy:   {accuracy_score(y_train, y_train_pred):.3f}')
print(f'validation accuracy: {accuracy_score(y_val, y_val_pred):.3f}')

Our first two rows in the results log: the same model, scored on the training set and on the validation set.

In [ ]:
log_run(section='3b', model='KNN (k=1)', features='4 measurements', preprocessing='none',
        change='first model', scored_on='training', accuracy=accuracy_score(y_train, y_train_pred))
log_run(section='3b', model='KNN (k=1)', features='4 measurements', preprocessing='none',
        change='same model, scored on the validation set', scored_on='validation', accuracy=accuracy_score(y_val, y_val_pred))
show_log(new=2)

#### Question #1

<img src="QRCodes/13_01.png" width="140" align="right">

Training accuracy is exactly 1.000. Why?

- A) Every training penguin's nearest neighbor is itself
- B) The model learned the true rule for telling species apart
- C) The validation set leaked into the training data
- D) There are no Chinstraps in the training set

Poll: https://forms.gle/srQHLZvB92f6AmyH9

<div class="alert alert-warning">

**Training accuracy is too optimistic.** The validation accuracy (0.81) is the honest estimate, because the model never saw those penguins. Always judge a model on data it wasn't trained on.

</div>

<div class="alert alert-info">

**A6:** You'll see the same gap in 2c/2d, between training and test.

</div>

### 3c. Evaluate <a id="s3c"></a>

Accuracy is one number. `classification_report` breaks performance down by class.

<div class="alert alert-warning">

The order of the arguments matters: **true labels first, then predictions**. If you swap them, precision and recall swap too.

</div>

In [ ]:
print(classification_report(y_val, y_val_pred))

For each species:
- **precision**: of the penguins the model *predicted* to be this species, the fraction that really are. Low precision means many **false positives**.
- **recall** (sensitivity): of the penguins that really *are* this species, the fraction the model found. Low recall means many **false negatives**.
- **f1-score**: the harmonic mean of precision and recall (best 1, worst 0)
- **support**: how many validation-set penguins are actually this species

And the summary rows:
- **accuracy**: fraction of all validation penguins predicted correctly
- **macro avg**: unweighted mean across the species (each species counts equally)
- **weighted avg**: mean across species, weighted by support

The **confusion matrix** shows where the errors are. Rows are the true species and columns are the predicted species, so correct predictions are on the diagonal.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, y_val_pred)
plt.show()

#### Question #2

<img src="QRCodes/13_01.png" width="140" align="right">

The precision for Chinstrap is 0.60. What does that mean?

- A) The model found 60% of the actual Chinstraps in the validation set
- B) When the model predicts Chinstrap, it's right 60% of the time
- C) 60% of the validation set is Chinstrap
- D) The model's accuracy on Chinstraps is 60%

Poll: https://forms.gle/srQHLZvB92f6AmyH9

<div class="alert alert-warning">

Remember the class imbalance from section 2? Chinstrap, the smallest class, has the lowest precision and f1-score. Fewer examples to learn from usually means worse performance on that class.

</div>

<div class="alert alert-info">

**A6:** Question 2h asks about this.

</div>

## 4. Scaling: Units Matter for KNN <a id="s4"></a>

KNN decides who's "nearest" using the distance between penguins' measurements. That's where the units come in.

### 4a. Look at the Scales <a id="s4a"></a>

In [ ]:
X_train.describe().loc[['mean', 'std']].round(1)

#### Question #3

<img src="QRCodes/13_01.png" width="140" align="right">

KNN uses the distance between measurements. On these raw values, which feature mostly decides who a penguin's nearest neighbors are?

- A) `bill_length_mm`
- B) `bill_depth_mm`
- C) `flipper_length_mm`
- D) `body_mass_g`

Poll: https://forms.gle/srQHLZvB92f6AmyH9

### 4b. Fit the Scaler on Training Data Only <a id="s4b"></a>

`StandardScaler` learns each feature's mean and standard deviation when you `.fit()` it. **Fit it on the training data only**, then use that same fit to `.transform()` any held-out data. If you fit on all the data, information about the held-out penguins leaks into training.

<div class="alert alert-info">

**A6:** This is Part 3 (3a). There, the held-out set is called `X_test`, so you'll name the result `X_test_scaled`.

</div>

In [ ]:
# create a StandardScaler and fit it on the TRAINING data only,
# then use it to transform both X_train and X_val
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit and transform the training data
X_val_scaled = scaler.transform(X_val)          # transform only; never refit on held-out data

In [ ]:
# every feature now has mean 0 and standard deviation 1 (in the training data)
pd.DataFrame(X_train_scaled, columns=features).describe().loc[['mean', 'std']].round(2)

### 4c. KNN on Scaled Data <a id="s4c"></a>

In [ ]:
knn_scaled = KNeighborsClassifier(n_neighbors=1)
knn_scaled.fit(X_train_scaled, y_train)

y_val_pred = knn_scaled.predict(X_val_scaled)

print(f'validation accuracy: {accuracy_score(y_val, y_val_pred):.3f}')
print(classification_report(y_val, y_val_pred))

In [ ]:
log_run(section='4c', model='KNN (k=1)', features='4 measurements', preprocessing='z-score',
        change='scaled the features', scored_on='validation', accuracy=accuracy_score(y_val, y_val_pred))
show_log()

<div class="alert alert-success">

**Decision: scale the features.** Validation accuracy went from 0.81 to 0.99 just by putting the features on the same scale, and we made that choice without touching the test set.

</div>

#### Question #4

<img src="QRCodes/13_01.png" width="140" align="right">

In A6 you'll see the opposite: on the forest cover data, scaling makes KNN *worse* (about 0.97 → 0.90). What's the most likely explanation?

- A) There's a bug, since scaling always helps KNN
- B) The forest cover test set is too small to tell
- C) The features measured in big units (elevation, distances in meters) are also the most informative, and scaling took away their extra weight
- D) Scaling changes the labels

Poll: https://forms.gle/srQHLZvB92f6AmyH9

## 5. Pipelines, Tuning & Comparing Models <a id="s5"></a>

### 5a. Pipelines <a id="s5a"></a>

Fitting a scaler on the training data, transforming the held-out data, then fitting the model is easy to get wrong. A `Pipeline` chains the steps into one object that you `.fit()` and `.predict()` like a model. Each step gets a name.

<div class="alert alert-info">

**A6:** 4a builds a scaler → PCA → KNN pipeline the same way.

</div>

In [ ]:
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=1))
])

knn_pipe.fit(X_train, y_train)        # fits the scaler on X_train, scales it, then fits KNN
y_val_pred = knn_pipe.predict(X_val)  # scales X_val with the training fit, then predicts

print(f'validation accuracy: {accuracy_score(y_val, y_val_pred):.3f}')

Same result as 4c, in one step, with no chance of accidentally refitting on held-out data.

### 5b. Training, Validation & Test <a id="s5b"></a>

Every time we make a **choice** (a hyperparameter like `k`, whether to scale, which model to use), we score the options on data the model wasn't trained on. That's the validation data's job. The test set stays locked away so that, at the end, it's still an honest estimate of performance on new data.

| Data | Used to | When |
|---|---|---|
| **Training** | fit the model | over and over |
| **Validation** | compare options: hyperparameter values, preprocessing, different models | every time we make a choice |
| **Test** | estimate how the final model will do on new data | **once**, at the very end |

So far our validation data has been a single set of 67 penguins. With a small dataset, one validation set is noisy: a different random split would give a somewhat different score (see [9b](#s9b)). **Cross-validation** is usually better: put the training and validation data back together (`X_dev`, everything except the test set), split it into 5 folds, train on 4 and score on the 5th, and repeat so that each fold takes a turn as the validation set.

In [ ]:
# 5 validation scores: one per fold, each from a model trained on the other 4 folds
scores = cross_val_score(knn_pipe, X_dev, y_dev, cv=5)

print(scores.round(3))
print(f'mean CV accuracy: {scores.mean():.3f}')

A new kind of score, so the *Change* column starts over for cross-validation:

In [ ]:
run_scaled_cv = log_run(section='5b', model='KNN (k=1)', features='4 measurements', preprocessing='z-score (Pipeline)',
                        change='validated with 5-fold CV instead of one validation set', scored_on='5-fold CV',
                        accuracy=scores.mean())
show_log()

Because the scaler is inside the pipeline, `cross_val_score` refits it within each fold, on that fold's training portion only. No leakage, even inside cross-validation.

### 5c. Tuning k with GridSearchCV <a id="s5c"></a>

Why `k = 1`? `k` is a **hyperparameter**: we choose it, the model doesn't learn it. We could loop over values of `k` and cross-validate each one:

In [ ]:
for k in [1, 3, 5, 7, 9, 11, 15]:
    knn_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(knn_pipe, X_dev, y_dev, cv=5)
    print(f'k = {k:>2}: mean CV accuracy = {scores.mean():.3f}')

`GridSearchCV` does that loop for you: it tries every value in a grid, cross-validates each one, and then refits the best one on all of the data you gave it (here, `X_dev`).

To set a hyperparameter of a pipeline step, use `<step name>__<parameter>` (two underscores): here, `knn__n_neighbors`.

In [ ]:
params = {'knn__n_neighbors': [1, 3, 5, 7, 9, 11, 15]}

searcher = GridSearchCV(knn_pipe, param_grid=params, cv=5)
searcher.fit(X_dev, y_dev)

print('best k:', searcher.best_params_)
print(f'best mean CV accuracy: {searcher.best_score_:.3f}')

In [ ]:
# the cross-validation results for every k
pd.DataFrame(searcher.cv_results_)[['param_knn__n_neighbors', 'mean_test_score', 'rank_test_score']]

<div class="alert alert-warning">

`scikit-learn` calls each fold's held-out score a "test" score, as in `mean_test_score`. Here, those folds are playing the role of **validation** data. Our real test set is still locked away.

</div>

In [ ]:
tuned_knn = describe(searcher.best_estimator_.named_steps['knn'], searcher.best_params_)

run_tuned_knn = log_run(section='5c', model=tuned_knn, features='4 measurements', preprocessing='z-score (Pipeline)',
                        change='tuned k with GridSearchCV (best of 7 values)', scored_on='5-fold CV',
                        accuracy=searcher.best_score_)
show_log()

<div class="alert alert-success">

**Decision: `k = 5`**, the best mean CV accuracy (0.989) of the 7 values we tried.

</div>

<div class="alert alert-warning">

Normally this is where you'd be tempted to check the test set. Don't: we haven't finished making choices.

</div>

Next, should we even be using KNN?

#### Question #5

<img src="QRCodes/13_01.png" width="140" align="right">

Which data did `GridSearchCV` use to choose k?

- A) The test set
- B) The training and validation data (everything except the test set), split into 5 folds
- C) All 333 penguins
- D) Only the 67-penguin validation set from 3a

Poll: https://forms.gle/srQHLZvB92f6AmyH9

### 5d. Comparing Different Models <a id="s5d"></a>

Choosing between KNN, an SVM, and a decision tree is just another choice, so it's made the same way: with cross-validation, never the test set. Each model gets its own small grid of hyperparameters, and the model itself becomes something `GridSearchCV` searches over. (Scaling doesn't matter for the tree, but it doesn't hurt.)

In [ ]:
model_pipe = Pipeline([('scaler', StandardScaler()),
                       ('model', KNeighborsClassifier())])   # 'model' is a placeholder; the grid swaps it out

search_space = [{'model': [KNeighborsClassifier()],
                 'model__n_neighbors': [1, 3, 5, 7, 9, 11, 15]},
                {'model': [SVC()],
                 'model__kernel': ['linear', 'rbf'],
                 'model__C': [0.1, 1, 10, 100]},
                {'model': [DecisionTreeClassifier(random_state=42)],
                 'model__max_depth': [1, 2, 3, 4, 5, None]}]

model_search = GridSearchCV(model_pipe, search_space, cv=5)
model_search.fit(X_dev, y_dev)

print(model_search.best_params_)

In [ ]:
# the best settings (and mean CV accuracy) for each kind of model, in the order they appear in the search space
results = pd.DataFrame(model_search.cv_results_)
results['model_type'] = results['param_model'].apply(lambda m: type(m).__name__)
best_rows = results.loc[results.groupby('model_type', sort=False)['mean_test_score'].idxmax()]

run_by_type = {}
for _, row in best_rows.iterrows():
    run_by_type[row['model_type']] = log_run(
        section='5d', model=describe(row['param_model'], row['params']), features='4 measurements',
        preprocessing='z-score (Pipeline)', change='compared model types with GridSearchCV',
        scored_on='5-fold CV', accuracy=row['mean_test_score'],
        compare_to=run_tuned_knn)   # every model type vs. the tuned KNN from 5c
show_log(new=len(best_rows))

<div class="alert alert-success">

**Decision: drop the decision tree; keep KNN and the SVM.** KNN and the SVM tie exactly (they get the same penguins wrong across the 5 folds), and the tree is about 3 percentage points behind.

</div>

<div class="alert alert-warning">

`GridSearchCV` breaks ties by taking whichever comes first in the search space, which is why `best_params_` says KNN. That doesn't make KNN better. With 266 penguins, small differences in CV accuracy aren't meaningful on their own. When models are close, it's reasonable to prefer the simpler or more interpretable one, or the one that fits the rest of your problem better.

</div>

The test set is *still* locked: we have one more choice to make. So far we've ignored the categorical features.

## 6. Categorical Features: One-Hot Encoding <a id="s6"></a>

`island` and `sex` are categorical. Models need numbers, but coding categories as 1, 2, 3 invents an order that isn't there: if red = 3, blue = 2, green = 1, the model "learns" that red > blue, and that blue is closer to green than red is.

**One-hot encoding** gives a category with N levels N columns of 0s and 1s, with exactly one 1 per row: red = [1, 0, 0], blue = [0, 1, 0], green = [0, 0, 1]. No order, and every category is equally different from every other.

<div class="alert alert-info">

**A6:** The soil type and wilderness area columns are already one-hot encoded.

</div>

In [ ]:
# just to see what the encoding looks like; in a model, the encoder goes inside the Pipeline (section 7)
# so that it's fit on the training data only
onehot = OneHotEncoder()
encoded = onehot.fit_transform(penguins[['island', 'sex']]).toarray()

pd.DataFrame(encoded, columns=onehot.get_feature_names_out()).head()

## 7. A Template for Your Project <a id="s7"></a>

Putting it all together: one-hot encode the categorical features, z-score the numeric ones, and tune and compare models, all in one `Pipeline` with `GridSearchCV`.

We'll add `sex` but leave out `island`.

<div class="alert alert-warning">

**Why leave out `island`?** Island isn't a body measurement, and it nearly gives the answer away: in this data, Chinstraps live only on Dream, Gentoos only on Biscoe, and Torgersen has only Adelies. A model with `island` would mostly be learning where penguins were sampled.

</div>

KNN and the SVM tied in [5d](#s5d), and the tree was behind, so we carry both KNN and the SVM forward and let cross-validation choose between them now that `sex` is included. The **support vector machine (SVM)** classifier, `SVC`, draws the boundary between classes that leaves the widest possible margin. Its main hyperparameters:
- `kernel`: `'linear'` for a straight boundary, `'rbf'` for a curved one
- `C`: how hard the margin is. Large `C` allows few training points inside the margin (a hard margin, which can overfit). Small `C` allows more (a soft margin).
- `gamma` (rbf only): how wiggly the boundary can be

How SVMs work is in On Your Own ([9d](#s9d)–[9f](#s9f)).

<div class="alert alert-warning">

**This is a template, not something to copy and paste.** Your data may need different preprocessing, a different model, or a different grid. Make sure you understand each step.

</div>

In [ ]:
# predict species from the body measurements plus sex (island left out; see above)
X_all = penguins.drop(columns=['species', 'island'])
y_all = penguins['species']

# same test_size and random_state as 3a, so this locks away the same test penguins
X_all_dev, X_all_test, y_all_dev, y_all_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=30)
assert (X_all_test.index == X_test.index).all()

# which columns are categorical and which are numeric
categorical_columns = ['sex']
numerical_columns = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

# step 1: one-hot the categorical columns and z-score the numeric ones
preprocessor = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
    ('zscore', StandardScaler(), numerical_columns)])

# step 2: chain preprocessing and the model into one pipeline
pipe = Pipeline([('make_features', preprocessor),
                 ('model', SVC())])   # 'model' is a placeholder; the grid swaps it out

# step 3: the models and hyperparameters to search; each gets only the hyperparameters it uses
search_space = [{'model': [KNeighborsClassifier()],
                 'model__n_neighbors': [1, 3, 5, 7, 9, 11, 15]},
                {'model': [SVC()],
                 'model__kernel': ['linear'],
                 'model__C': np.logspace(-3, 2, 11)},          # 11 values from 10^-3 to 10^2
                {'model': [SVC()],
                 'model__kernel': ['rbf'],
                 'model__gamma': np.logspace(-3, 2, 11),
                 'model__C': np.logspace(-3, 2, 11)}]

# step 4: cross-validate every combination on everything except the test set (takes a few seconds)
best_model = GridSearchCV(pipe, search_space, cv=5)
best_model.fit(X_all_dev, y_all_dev)

print(best_model.best_params_)
print(f'best mean CV accuracy: {best_model.best_score_:.3f}')

In [ ]:
# the best settings for each kind of model; each is compared to the same model type without sex, from 5d
template_results = pd.DataFrame(best_model.cv_results_)
template_results['model_type'] = template_results['param_model'].apply(lambda m: type(m).__name__)
template_best = template_results.loc[template_results.groupby('model_type', sort=False)['mean_test_score'].idxmax()]

for _, row in template_best.iterrows():
    log_run(section='7', model=describe(row['param_model'], row['params']), features='4 measurements + sex',
            preprocessing='one-hot + z-score', change='added sex; tuned with GridSearchCV',
            scored_on='5-fold CV', accuracy=row['mean_test_score'],
            compare_to=run_by_type[row['model_type']])
show_log(new=len(template_best))

<div class="alert alert-warning">

With `sex` included, the SVM edges out KNN: 0.996 vs. 0.992 mean CV accuracy. That's a difference of **one penguin** out of 266, so it's essentially still a tie. When models are this close, the CV score alone shouldn't decide. Other things to weigh:

- **Simplicity and interpretability.** This SVM is linear: it learns one weight per feature, which you can inspect to see what drives its predictions. KNN has no weights to look at; each prediction is "these were the closest training penguins."
- **Cost of making predictions.** KNN keeps every training point and compares each new point to all of them. That's why A6's prediction cells are slow with 581,012 forest plots. Once an SVM is fit, predicting is fast.
- **What the model is for.** If you need to explain individual predictions, "here are the 5 most similar penguins" can be easier to explain than a weighted boundary.

</div>

<div class="alert alert-success">

**Decision: the linear SVM is our final model.** The CV score and most of the tiebreakers point to it. It's `best_model`, which `GridSearchCV` has already refit on all of the non-test data.

</div>

### 7a. Open the Test Set, Once <a id="s7a"></a>

That was our last choice. This is our final model, and the cross-validation score above is our estimate of how well it will do on new penguins. Now, for the first and only time, we score on the test set: 67 penguins that played no part in any decision we made.

In [ ]:
final_model_name = describe(best_model.best_params_['model'], best_model.best_params_)
print('final model:', final_model_name)

y_all_test_pred = best_model.predict(X_all_test)

print(f'CV estimate:   {best_model.best_score_:.3f}')
print(f'test accuracy: {accuracy_score(y_all_test, y_all_test_pred):.3f}')
print()
print(classification_report(y_all_test, y_all_test_pred))
ConfusionMatrixDisplay.from_predictions(y_all_test, y_all_test_pred)
plt.show()

In [ ]:
log_run(section='7a', model=final_model_name, features='4 measurements + sex', preprocessing='one-hot + z-score',
        change='final model: scored on the test set, once', scored_on='test',
        accuracy=accuracy_score(y_all_test, y_all_test_pred))
show_log()

Did cross-validation give us a good estimate? Compare the CV estimate to the test accuracy. With only 67 test penguins, one penguin is worth 1.5 percentage points, so expect them to be close but not identical.

<div class="alert alert-warning">

Whatever the test score is, **we report it as is**. Going back now to try other models and checking the test set again would turn it into a second validation set.

</div>

## 8. Summary: The Steps <a id="s8"></a>

1. **Split first.** Lock away a test set. Use the rest for training and validation (a single validation set, or cross-validation).
2. **Decide on preprocessing**: scale numeric features (for distance-based models), one-hot encode categorical ones.
3. **Fit preprocessing on the training data only.** A `Pipeline` does this for you, even inside cross-validation.
4. **Make every choice with validation data**: hyperparameters, preprocessing, and which model (`GridSearchCV`).
5. **Score the final model on the test set once**, at the end: `classification_report` and the confusion matrix, by class and not just overall accuracy. Compare it to your CV estimate.
6. **Ask who the errors fall on.** Which classes (or people) does the model get wrong?

Here's everything we tried today. Look down the *Scored on* column: every choice (scaling, `k`, which model, adding `sex`) was made with validation or cross-validation scores, and the test set appears exactly **once**, at the very end.

In [ ]:
show_log(new=0)

## 9. On Your Own <a id="s9"></a>

We won't get to these in lecture, but they're worth working through before A6 and your project.

### 9a. A Different Model: Decision Trees <a id="s9a"></a>

A decision tree predicts with a series of yes/no questions about one feature at a time ("is the flipper longer than 206 mm?"). Because each question looks at a single feature, trees **don't need scaling**.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)   # the raw, unscaled measurements

tree_train_acc = accuracy_score(y_train, tree.predict(X_train))
tree_cv_acc = cross_val_score(tree, X_dev, y_dev, cv=5).mean()

print(f'training accuracy: {tree_train_acc:.3f}')
print(f'mean CV accuracy:  {tree_cv_acc:.3f}')

In [ ]:
log_run(section='9a', model='Decision tree (no max_depth)', features='4 measurements', preprocessing='none',
        change='switched to a decision tree', scored_on='training', accuracy=tree_train_acc)  # vs. run 1, the KNN
log_run(section='9a', model='Decision tree (no max_depth)', features='4 measurements', preprocessing='none',
        change='no max_depth, no scaling', scored_on='5-fold CV', accuracy=tree_cv_acc,
        compare_to=run_by_type['DecisionTreeClassifier'])   # vs. the tuned tree from 5d
show_log(new=2)

<div class="alert alert-warning">

An unlimited tree keeps splitting until it gets every training penguin right, so its training accuracy is perfect. The cross-validation accuracy is the honest comparison.

</div>

Many species have differently sized males and females, so a small male of one species can measure like a large female of another. Does adding `sex` help the tree? Compare with cross-validation (the template's preprocessor one-hot encodes `sex`):

In [ ]:
tree_with_sex = Pipeline([('make_features', preprocessor),
                          ('model', DecisionTreeClassifier(random_state=42))])

cv_with_sex = cross_val_score(tree_with_sex, X_all_dev, y_all_dev, cv=5).mean()

print(f'without sex: mean CV accuracy = {tree_cv_acc:.3f}')
print(f'with sex:    mean CV accuracy = {cv_with_sex:.3f}')

In [ ]:
log_run(section='9a', model='Decision tree (no max_depth)', features='4 measurements + sex', preprocessing='one-hot + z-score',
        change='added sex', scored_on='5-fold CV', accuracy=cv_with_sex)
show_log()

Try it: what does `max_depth` do to training accuracy and to CV accuracy? Use `GridSearchCV` to choose it.

<div class="alert alert-warning">

We already used our test set in 7a, so there's no test score for this one. In a real project, all of this exploring would happen *before* you open the test set.

</div>

### 9b. How Noisy Is One Validation Set? <a id="s9b"></a>

In sections 3–5a we used a single validation set of 67 penguins. How much does its score depend on *which* 67 penguins happened to land in it? Repeat the training/validation split 20 times with different random seeds, scoring the same model (scaled KNN, `k = 5`) each time:

In [ ]:
model = Pipeline([('scaler', StandardScaler()),
                  ('knn', KNeighborsClassifier(n_neighbors=5))])

val_scores = []
for seed in range(20):
    X_tr, X_va, y_tr, y_va = train_test_split(X_dev, y_dev, test_size=0.25, random_state=seed)
    model.fit(X_tr, y_tr)
    val_scores.append(accuracy_score(y_va, model.predict(X_va)))

print(f'validation accuracy over 20 splits: min {min(val_scores):.3f}, max {max(val_scores):.3f}')

cv_scores = cross_val_score(model, X_dev, y_dev, cv=5)
print(f'5-fold CV accuracy: {cv_scores.mean():.3f}')

<div class="alert alert-warning">

The same model gets noticeably different validation scores depending on the split. With small data, two options can swap places just by luck. Cross-validation averages over 5 validation sets that together use every penguin once, so it's more stable. With a large dataset (like A6's 581,012 forest plots), a single validation set is fine.

</div>

### 9c. How Should You Split? Time Matters <a id="s9c"></a>

<div class="alert alert-warning">

`train_test_split` shuffles the rows, which assumes new data will look like the data you have. Often you'll use a model on the **future**: next quarter's survey responses, next year's patients. Then a random split mixes past and future, and test performance will look better than it really is.

</div>

A more honest test for a model that will be used on future data: train on earlier data and test on the most recent data. For example, train on survey responses from past quarters and test on the most recent quarter.

Think about it: Is your project's data collected over time? If you built a model, how would you split it?

### 9d. SVMs: Maximizing the Margin <a id="s9d"></a>

A support vector machine is a linear classifier: it separates classes with a line (or, with more features, a plane). Many lines separate these two groups. Rather than a zero-width line, imagine a *margin* around each line, out to the nearest point:

In [ ]:
from sklearn.datasets import make_blobs

X_blobs, y_blobs = make_blobs(n_samples=50, centers=2,
                              random_state=0, cluster_std=0.60)

xfit = np.linspace(-1, 3.5)
sns.scatterplot(x=X_blobs[:, 0], y=X_blobs[:, 1], hue=y_blobs, s=75)

for m, b, d in [(1, 0.65, 0.33), (0.5, 1.6, 0.55), (-0.2, 2.9, 0.2)]:
    yfit = m * xfit + b
    plt.plot(xfit, yfit, '-k')
    plt.fill_between(xfit, yfit - d, yfit + d, edgecolor='none',
                     color='#AAAAAA', alpha=0.4)

plt.xlim(-1, 3.5)
plt.show()

An SVM chooses the line with the **widest margin**. Let's fit one, with a linear kernel and a very large `C` (a hard margin):

In [ ]:
model = SVC(kernel='linear', C=1E10)
model.fit(X_blobs, y_blobs)

A function to plot an SVM's decision boundary and margins:

In [ ]:
def plot_svc_decision_function(model, ax=None, plot_support=True):
    """Plot the decision function for a 2D SVC"""
    if ax is None:
        ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    # create grid to evaluate model
    x = np.linspace(xlim[0], xlim[1], 30)
    y = np.linspace(ylim[0], ylim[1], 30)
    Y, X = np.meshgrid(y, x)
    xy = np.vstack([X.ravel(), Y.ravel()]).T
    P = model.decision_function(xy).reshape(X.shape)

    # plot decision boundary and margins
    ax.contour(X, Y, P, colors='k',
               levels=[-1, 0, 1], alpha=0.5,
               linestyles=['--', '-', '--'])

    # plot support vectors
    if plot_support:
        ax.scatter(model.support_vectors_[:, 0],
                   model.support_vectors_[:, 1],
                   s=300, linewidth=3, facecolors='none', edgecolor='k')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
sns.scatterplot(x=X_blobs[:, 0], y=X_blobs[:, 1], hue=y_blobs, s=75)
plot_svc_decision_function(model)
plt.show()

The solid line is the boundary, and the dashed lines are the edges of the margin. The circled points touch the margin. They're the **support vectors**, which give the method its name, and they're stored in `.support_vectors_`:

In [ ]:
model.support_vectors_

Only the support vectors determine the fit. Points farther from the margin, on the correct side, don't change it. To see this, fit the model to the first 30, 60, and 120 points of a larger dataset:

In [ ]:
def plot_svm(N=10, ax=None):
    X, y = make_blobs(n_samples=200, centers=2,
                      random_state=0, cluster_std=0.60)
    X = X[:N]
    y = y[:N]
    model = SVC(kernel='linear', C=1E10)
    model.fit(X, y)

    ax = ax or plt.gca()
    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, s=75, ax=ax)
    ax.set_xlim(-1, 4)
    ax.set_ylim(-1, 6)
    plot_svc_decision_function(model, ax)

fig, ax = plt.subplots(1, 3, figsize=(16, 6))
fig.subplots_adjust(left=0.0625, right=0.95, wspace=0.1)
for axi, N in zip(ax, [30, 60, 120]):
    plot_svm(N, axi)
    axi.set_title('N = {0}'.format(N))
plt.show()

Going from 30 to 60 points changed the fit. Going from 60 to 120 didn't, because the support vectors stayed the same. This insensitivity to distant points is one of the strengths of SVMs.

### 9e. Kernel SVMs <a id="s9e"></a>

Some data can't be separated by any straight line:

In [ ]:
from sklearn.datasets import make_circles

X_circles, y_circles = make_circles(100, factor=.1, noise=.1, random_state=0)

sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=y_circles, s=75)
plt.xlabel('x1')
plt.ylabel('x2')
plt.show()

One fix is to add a new feature that makes the classes separable. Here, a *radial basis function* centered on the middle clump: it's large near the center and small far away.

In [ ]:
phi = np.exp(-(X_circles ** 2).sum(1))

Adding `phi` as a third dimension:

In [ ]:
from matplotlib.colors import ListedColormap
cmap = ListedColormap(sns.color_palette()[:2])

with sns.plotting_context('notebook'):
    ax = plt.subplot(projection='3d')
    ax.scatter3D(X_circles[:, 0], X_circles[:, 1], phi, c=y_circles, s=7, cmap=cmap)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.set_zlabel(r'$\phi(\mathbf{x})$')
    plt.show()

Now a flat plane (at about `phi` = 0.7) separates the classes.

Here we had to choose the new feature, and center it in the right place. Instead, an SVM can effectively compute a basis function centered at *every* point and let the fit sort out which matter. This is a **kernel transformation**. Building all those features explicitly would be expensive, but the [kernel trick](https://en.wikipedia.org/wiki/Kernel_trick) lets the SVM fit as if it had them without ever computing them.

In `scikit-learn`, use an RBF (radial basis function) kernel by changing the `kernel` hyperparameter:

In [ ]:
clf = SVC(kernel='rbf', C=1)
clf.fit(X_circles, y_circles)

sns.scatterplot(x=X_circles[:, 0], y=X_circles[:, 1], hue=y_circles, s=75)
plot_svc_decision_function(clf)
plt.show()

The kernel SVM learns a curved boundary. Kernels are a common way to turn fast linear methods into fast nonlinear ones.

### 9f. Softening the Margin: C <a id="s9f"></a>

So far the classes were far apart, with plenty of room for a margin. Real data are usually messier. Here the two groups come close, and a few points crowd the space between them:

In [ ]:
X_overlap, y_overlap = make_blobs(n_samples=100, centers=2,
                                  random_state=0, cluster_std=0.8)
sns.scatterplot(x=X_overlap[:, 0], y=X_overlap[:, 1], hue=y_overlap, s=75)
plt.show()

The hyperparameter `C` lets some points fall inside the margin when that gives a better overall fit:
- **large `C`**: a hard margin. Few points are allowed inside it.
- **small `C`**: a soft margin. It can widen to include some points.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
fig.subplots_adjust(left=0.0625, right=0.95, wspace=0.1)

for axi, C in zip(ax, [10.0, 0.1]):
    model = SVC(kernel='linear', C=C).fit(X_overlap, y_overlap)
    sns.scatterplot(x=X_overlap[:, 0], y=X_overlap[:, 1], hue=y_overlap, s=75, ax=axi)
    plot_svc_decision_function(model, axi)
    axi.set_title('C = {0:.1f}'.format(C), size=14)
plt.show()

<div class="alert alert-warning">

The best `C` depends on your data, so choose it (along with the kernel and `gamma`) with cross-validation, as in the [project template](#s7).

</div>